In [17]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
import gensim
from gensim import corpora
from gensim.models import LdaModel

nltk.download("stopwords", quiet=True)
STOPWORDS = set(stopwords.words("english"))

In [18]:
CUSTOM_STOPWORDS = STOPWORDS.union({
    "amp", "dont", "cant", "wont", "im", "ive", "youre", "thats",
    "get", "got", "like", "would", "really", "one", "know", "want",
    "go", "going", "still", "much", "lol"
})

def clean_for_topics(text):
    if not isinstance(text, str):
        return []
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"&amp;?", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    words = text.split()
    words = [w for w in words if w not in CUSTOM_STOPWORDS and len(w) > 2]
    return words

df["tokens"] = df["text"].apply(clean_for_topics)
df = df[df["tokens"].map(len) > 0].reset_index(drop=True)

print(df.shape)

(49616, 7)


In [19]:
dictionary = corpora.Dictionary(df["tokens"])
dictionary.filter_extremes(no_below=5, no_above=0.5)  # remove rare and overly common words

corpus = [dictionary.doc2bow(tokens) for tokens in df["tokens"]]

print("Vocabulary size:", len(dictionary))
print("Number of documents:", len(corpus))

Vocabulary size: 6137
Number of documents: 49616


In [20]:
NUM_TOPICS = 6

lda_model = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=NUM_TOPICS,
    random_state=42,
    passes=10,
    alpha="auto"
)

for idx, topic in lda_model.print_topics(num_words=8):
    print(f"Topic {idx}: {topic}")

Topic 0: 0.054*"day" + 0.038*"time" + 0.029*"night" + 0.024*"last" + 0.020*"great" + 0.018*"fun" + 0.016*"sleep" + 0.015*"getting"
Topic 1: 0.024*"didnt" + 0.021*"sorry" + 0.019*"make" + 0.015*"never" + 0.013*"show" + 0.013*"take" + 0.013*"watch" + 0.012*"maybe"
Topic 2: 0.033*"haha" + 0.029*"right" + 0.027*"bad" + 0.021*"watching" + 0.018*"hey" + 0.014*"movie" + 0.012*"amazing" + 0.010*"saw"
Topic 3: 0.044*"love" + 0.029*"new" + 0.025*"miss" + 0.020*"twitter" + 0.020*"happy" + 0.017*"wish" + 0.017*"come" + 0.015*"could"
Topic 4: 0.039*"good" + 0.029*"work" + 0.028*"today" + 0.024*"back" + 0.021*"see" + 0.021*"well" + 0.018*"think" + 0.017*"home"
Topic 5: 0.023*"wait" + 0.023*"way" + 0.020*"hate" + 0.017*"awesome" + 0.017*"sick" + 0.016*"wanna" + 0.013*"yay" + 0.012*"ever"


In [21]:
import joblib

lda_model.save("../trained_models/lda_model.gensim")
dictionary.save("../trained_models/lda_dictionary.gensim")

topic_labels = {
    0: "Daily Routine / Time",
    1: "Apology / Hesitation",
    2: "Entertainment / Movies",
    3: "Affection / Wishing",
    4: "Work / Daily Life",
    5: "Mixed Mood / Reactions"
}

joblib.dump(topic_labels, "../trained_models/lda_topic_labels.pkl")

print("Saved LDA model, dictionary, and topic labels.")

Saved LDA model, dictionary, and topic labels.
